In [1]:
import sys
path='/home/tomas/Ulmer-Berechnung/alps2qutipplus-april/alps2qutipplus-main/'

sys.path.insert(1, path) 

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import qutip as qutip
import scipy.linalg as linalg

from alpsqutip import (build_system, list_models_in_alps_xml,
                       list_geometries_in_alps_xml, graph_from_alps_xml,
                       model_from_alps_xml,
                       restricted_maxent_toolkit as me)

from alpsqutip.operators.states.utils import safe_exp_and_normalize ## function used to safely and robustly map K-states to states
from alpsqutip.operators.states.meanfield import (one_body_from_qutip_operator, 
                                                  project_operator_to_m_body, 
                                                  project_to_n_body_operator,
                                                  project_qutip_operator_to_m_body,
                                                  project_meanfield,
                                                  self_consistent_project_meanfield,
                                                  self_consistent_quadratic_mfa)

from alpsqutip.scalarprod import fetch_covar_scalar_product, orthogonalize_basis

from alpsqutip.operators import (
    ScalarOperator,
    LocalOperator,
    OneBodyOperator,
    Operator,
    ProductOperator,
    ScalarOperator,
    SumOperator,
    QutipOperator
)

from alpsqutip.scalarprod import gram_matrix
from alpsqutip.operators.states.gibbs import GibbsDensityOperator

In [3]:
params={}

params['size']=6
params['Jx']=1.; params['Jy'] = .75*params['Jx']; params['Jz']=1.05*params['Jx']

from scipy.optimize import root, fsolve
Ffactor=np.real(max(np.roots(np.poly1d([1, 0, -(params['Jx']*params['Jy']+params['Jx']*params['Jy']+params['Jy']*params['Jz']), 
                           -2*params['Jx']*params['Jy']*params['Jz']]))))
chi_y=fsolve(lambda x,y: x*np.arcsinh(x)-np.sqrt(x**2+1)-y, 1e-1, args=(0))[0]
vLR=4*Ffactor*chi_y

In [4]:
system=build_system(geometry_name= "open chain lattice",model_name="spin", 
                    L=params['size'], J=1)

sites=[s for s in system.sites]
sx_ops=[system.site_operator("Sx", '1[' + str(a) + ']') for a in range(len(system.sites))]
sy_ops=[system.site_operator("Sy", '1[' + str(a) + ']') for a in range(len(system.sites))]
sz_ops=[system.site_operator("Sz", '1[' + str(a) + ']') for a in range(len(system.sites))]

Sz = system.global_operator("Sz")
H = (
    params['Jx'] * sum(sx_ops[i] * sx_ops[i+1] for i in range(len(sites)-1)) +
    params['Jy'] * sum(sy_ops[i] * sy_ops[i+1] for i in range(len(sites)-1)) +
    params['Jz'] * sum(sz_ops[i] * sz_ops[i+1] for i in range(len(sites)-1))
)

idop = [system.site_operator('identity@1[' + str(a) + ']') for a in range(len(system.sites))]
idop = idop[0]*idop[1]#*idop[2]#*idop[3]#*idop[4]*idop[5]*idop[6]

loading model spin  over graph open chain lattice


In [21]:
HBB0=[idop, 
      system.site_operator('Sx', '1[0]'), 
      system.site_operator('Sy', '1[0]'), 
      system.site_operator('Sz', '1[0]')]

phi0 = np.array([.0, -.25, -.25, 
                 1])

K0 = me.k_state_from_phi_basis(phi0, HBB0)
sigma0 = GibbsDensityOperator(-K0)
phi0[0] = np.log(sigma0.tr())
K0 = me.k_state_from_phi_basis(phi0, HBB0)
sigma0 = GibbsDensityOperator(-K0)

[(sigma0 * op).tr() for op in sz_ops] 

[-0.22892976585114871,
 -3.469446951953614e-18,
 -1.0408340855860843e-17,
 0.0,
 -6.938893903907228e-18,
 0.0]

In [29]:
sp_local = fetch_covar_scalar_product(sigma0)

HBB_ell = me.build_hierarchical_basis(generator=H, 
                                      seed_op = K0,
                                      deep = 5)

gram_matrix = qutip.Qobj([[sp_local(op1, op2) for op1 in HBB_ell] for op2 in HBB_ell])


Hij_tensor_act = me.fn_hij_tensor_with_errors(generator=H, 
                                                        basis=HBB_ell,
                                                        sp=sp_local)[0] * 1j

mu=linalg.norm(gram_matrix.inv()*Hij_tensor_act)
mu

6.189787061681298

In [28]:
gram_matrix

Quantum object: dims = [[6], [6]], shape = (6, 6), type = oper, isherm = True
Qobj data =
[[ 2.81250000e-01  0.00000000e+00 -1.12373047e-01  0.00000000e+00
   1.37696606e-01  0.00000000e+00]
 [ 0.00000000e+00  1.12373047e-01  0.00000000e+00 -1.37696606e-01
  -3.86247024e-19  2.37072383e-01]
 [-1.12373047e-01  1.89735380e-19  1.37696606e-01  1.69406589e-19
  -2.37072383e-01 -2.02271468e-18]
 [ 5.42101086e-20 -1.37696606e-01  2.03287907e-19  2.37072383e-01
   4.35374935e-19 -4.96170471e-01]
 [ 1.37696606e-01 -2.71050543e-20 -2.37072383e-01  4.24363507e-19
   4.96170471e-01  2.00640929e-18]
 [-4.33680869e-19  2.37072383e-01 -2.13113490e-18 -4.96170471e-01
   2.18100396e-18  1.20502194e+00]]

In [12]:
qutip.Qobj(gram_matrix)

Quantum object: dims = [[6], [6]], shape = (6, 6), type = oper, isherm = True
Qobj data =
[[ 0.28125     0.         -0.11237305  0.          0.13769661  0.        ]
 [ 0.          0.11237305  0.         -0.13769661  0.          0.23707238]
 [-0.11237305  0.          0.13769661  0.         -0.23707238  0.        ]
 [ 0.         -0.13769661  0.          0.23707238  0.         -0.49617047]
 [ 0.13769661  0.         -0.23707238  0.          0.49617047  0.        ]
 [ 0.          0.23707238  0.         -0.49617047  0.          1.20502194]]

In [17]:
1j*qutip.Qobj(Hij_tensor_act)

Quantum object: dims = [[6], [6]], shape = (6, 6), type = oper, isherm = True
Qobj data =
[[0.+0.j         0.+0.11237305j 0.+0.j         0.-0.13769661j
  0.+0.j         0.+0.23707238j]
 [0.-0.11237305j 0.+0.j         0.+0.13769661j 0.+0.j
  0.-0.23707238j 0.+0.j        ]
 [0.+0.j         0.-0.13769661j 0.+0.j         0.+0.23707238j
  0.+0.j         0.-0.49617047j]
 [0.+0.13769661j 0.+0.j         0.-0.23707238j 0.+0.j
  0.+0.49617047j 0.+0.j        ]
 [0.+0.j         0.+0.23707238j 0.+0.j         0.-0.49617047j
  0.+0.j         0.+1.20502194j]
 [0.-0.23707238j 0.+0.j         0.+0.49617047j 0.+0.j
  0.-1.20502194j 0.+0.j        ]]